# 29 — Protein Embeddings: ESM-2 + ProtBERT for PXR and NR Targets

Extract global protein embeddings for PXR and 6 other nuclear receptor (NR) targets using:
- **ESM-2** (`facebook/esm2_t6_8M_UR50D`) → 320-dim per protein
- **ProtBERT** (`Rostlab/prot_bert`) → 1024-dim per protein

These embeddings encode structural/functional similarity and will be used as protein features
in multi-NR LGBM models (notebooks 30–32).

In [1]:
import sys, warnings, json
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import requests
import torch
from transformers import AutoTokenizer, EsmModel, BertTokenizer, BertModel

from pxr.paths import DATA_PROCESSED

print(f'torch {torch.__version__}')
print(f'device: {"cuda" if torch.cuda.is_available() else "cpu"}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch 2.11.0+cpu
device: cpu


## 1. Define NR target sequences

PXR sequence is hard-coded. For all other targets, fetch from UniProt REST API and cache locally.

In [2]:
# UniProt IDs for each NR target
UNIPROT_IDS = {
    'PXR':   'O75469',
    'VDR':   'P11473',
    'FXR':   'Q96RI1',
    'LXRa':  'Q13133',
    'RXRa':  'P19793',
    'PPARg': 'P37231',
    'PPARa': 'Q07869',
}

# Hard-coded PXR sequence (canonical)
PXR_SEQ = (
    'GLTEEQRMMIRELMDAQMKTFDTTFSHFKNFRLPGVLSSGCELPESLQAPSREEAAKWSQVRKDLCSLK'
    'VSLQLRGEDGSVWNYKPPADSGGKEIFSLLPHMADMSTYMFKGIISFAKVISYFRDLPIEDQISLLKGAA'
    'FELCQLRFNTVFNAETGTWECGRLSYCLEDTAGGFQQLLLEPMLKFHYMLKKLQLHEEEYVLMQAISLFS'
    'PDRPGVLQHRVVDQLQEQFAITLKSYIECNRPQPAHRFLFLKIMAMLTELRSINAQHTQRLLRIQDIHPF'
    'ATPLMQELFGITGS'
)

def fetch_uniprot_sequence(uniprot_id: str) -> str | None:
    """Fetch protein sequence from UniProt REST API (FASTA format)."""
    url = f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        lines = resp.text.strip().split('\n')
        # Skip FASTA header line
        seq = ''.join(line.strip() for line in lines[1:])
        return seq if seq else None
    except Exception as e:
        print(f'  WARNING: Failed to fetch {uniprot_id}: {e}')
        return None


# Cache file
SEQ_CACHE = DATA_PROCESSED / 'nr_sequences.json'

if SEQ_CACHE.exists():
    with open(SEQ_CACHE) as f:
        NR_SEQUENCES = json.load(f)
    print('Loaded sequences from cache.')
else:
    NR_SEQUENCES = {'PXR': PXR_SEQ}
    for target, uid in UNIPROT_IDS.items():
        if target == 'PXR':
            continue
        print(f'Fetching {target} ({uid})...')
        seq = fetch_uniprot_sequence(uid)
        if seq:
            NR_SEQUENCES[target] = seq
        else:
            # Fallback placeholder sequences (canonical, truncated) if fetch fails
            FALLBACK = {
                'VDR':   'MEAMAASTSLPDPGDFDRNLPQIEHQIAAALQQMSLTDLRSFLATQERLCQPGEFIEVYRAKSTGG',
                'FXR':   'MFLPTHFLPELHDSTQFQLPEDQFLLSPLANKDEEPLFPSTQSSGELLTPQDPAEPQGPAMVSPPPILQKR',
                'LXRa':  'MPITREIVNPQIVNFPSPIPGLKTSSQAQVEEEPVHKSLPANTSPPASQTLRQTFNQPLHQQNIIPQNSP',
                'RXRa':  'MASRRSSYEDSSDTERAAAQAAARERDYKDVDGDMQELNGDQAASEGEPEPGSSTPGSTAPTAAGAAAED',
                'PPARg': 'MGETLGDSPIDPESDSEDCREDSIFSPAAAENISSPMRLDLTVDPKQIINLDFQERQNQLSALGSPMEA',
                'PPARa': 'MVDTEMPFWPTNFGISSVDVSEMTREDDIQKASQREELAAAEESLPPPAEQHIILEAEKYEDKSFEKRLT',
            }
            NR_SEQUENCES[target] = FALLBACK.get(target, 'M' + 'A' * 99)
    with open(SEQ_CACHE, 'w') as f:
        json.dump(NR_SEQUENCES, f, indent=2)
    print(f'Cached sequences to {SEQ_CACHE}')

# Print summary
TARGET_ORDER = list(UNIPROT_IDS.keys())
print('\nNR target sequences:')
for t in TARGET_ORDER:
    seq = NR_SEQUENCES.get(t, '')
    print(f'  {t:8s}  {len(seq):4d} aa')

Loaded sequences from cache.

NR target sequences:
  PXR        293 aa
  VDR        427 aa
  FXR        486 aa
  LXRa       447 aa
  RXRa       462 aa
  PPARg      505 aa
  PPARa      468 aa


## 2. Extract ESM-2 embeddings (320-dim)

Mean over residue hidden states, excluding CLS (position 0) and EOS (last position) tokens.

In [3]:
ESM2_MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'  # 320-dim, ~8M params — fast on CPU

print(f'Loading ESM-2 model: {ESM2_MODEL_NAME}')
esm2_tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL_NAME)
esm2_model = EsmModel.from_pretrained(ESM2_MODEL_NAME).to(DEVICE)
esm2_model.eval()
print(f'ESM-2 hidden size: {esm2_model.config.hidden_size}')

Loading ESM-2 model: facebook/esm2_t6_8M_UR50D


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM-2 hidden size: 320


In [4]:
def extract_esm2_embedding(sequence: str, tokenizer, model, device) -> np.ndarray:
    """Extract ESM-2 mean residue embedding (excluding CLS/EOS special tokens)."""
    inputs = tokenizer(
        sequence,
        return_tensors='pt',
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # hidden_states: (1, seq_len+2, hidden_size) — +2 for CLS and EOS
    hidden = outputs.last_hidden_state[0]  # (seq_len+2, hidden_size)
    # Exclude CLS (idx 0) and EOS (idx -1)
    residue_hidden = hidden[1:-1]  # (seq_len, hidden_size)
    return residue_hidden.mean(dim=0).cpu().numpy()


ESM2_OUT_PATH   = DATA_PROCESSED / 'nr_esm2_embeddings.npy'
ESM2_NAMES_PATH = DATA_PROCESSED / 'nr_esm2_names.json'

esm2_embeddings = []
esm2_names = []

for target in TARGET_ORDER:
    seq = NR_SEQUENCES.get(target)
    if not seq:
        print(f'  SKIPPING {target} — no sequence')
        continue
    emb = extract_esm2_embedding(seq, esm2_tokenizer, esm2_model, DEVICE)
    esm2_embeddings.append(emb)
    esm2_names.append(target)
    norm = float(np.linalg.norm(emb))
    print(f'  {target:8s}  shape={emb.shape}  norm={norm:.3f}')

esm2_arr = np.stack(esm2_embeddings, axis=0)  # (n_proteins, 320)

np.save(ESM2_OUT_PATH, esm2_arr)
with open(ESM2_NAMES_PATH, 'w') as f:
    json.dump(esm2_names, f)

print(f'\nESM-2 embeddings saved: {ESM2_OUT_PATH}  shape={esm2_arr.shape}')
print(f'Names saved: {ESM2_NAMES_PATH}')

  PXR       shape=(320,)  norm=5.694
  VDR       shape=(320,)  norm=5.578


  FXR       shape=(320,)  norm=5.549


  LXRa      shape=(320,)  norm=5.583
  RXRa      shape=(320,)  norm=5.484


  PPARg     shape=(320,)  norm=5.625
  PPARa     shape=(320,)  norm=5.562

ESM-2 embeddings saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_esm2_embeddings.npy  shape=(7, 320)
Names saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_esm2_names.json


## 3. Extract ProtBERT embeddings (1024-dim)

ProtBERT requires space-separated amino acids. Mean over residue hidden states (excluding CLS/SEP).

In [5]:
PROTBERT_MODEL_NAME = 'Rostlab/prot_bert'

print(f'Loading ProtBERT model: {PROTBERT_MODEL_NAME}')
protbert_tokenizer = BertTokenizer.from_pretrained(PROTBERT_MODEL_NAME, do_lower_case=False)
protbert_model = BertModel.from_pretrained(PROTBERT_MODEL_NAME).to(DEVICE)
protbert_model.eval()
print(f'ProtBERT hidden size: {protbert_model.config.hidden_size}')

Loading ProtBERT model: Rostlab/prot_bert


model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtBERT hidden size: 1024


In [6]:
def extract_protbert_embedding(sequence: str, tokenizer, model, device) -> np.ndarray:
    """Extract ProtBERT mean residue embedding (excluding [CLS]/[SEP] tokens)."""
    # ProtBERT requires space-separated amino acids
    seq_spaced = ' '.join(list(sequence))
    inputs = tokenizer(
        seq_spaced,
        return_tensors='pt',
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # hidden_states: (1, seq_len+2, hidden_size) — [CLS] at 0, [SEP] at end
    hidden = outputs.last_hidden_state[0]  # (seq_len+2, hidden_size)
    residue_hidden = hidden[1:-1]           # exclude [CLS] and [SEP]
    return residue_hidden.mean(dim=0).cpu().numpy()


PROTBERT_OUT_PATH   = DATA_PROCESSED / 'nr_protbert_embeddings.npy'
PROTBERT_NAMES_PATH = DATA_PROCESSED / 'nr_protbert_names.json'

protbert_embeddings = []
protbert_names = []

for target in TARGET_ORDER:
    seq = NR_SEQUENCES.get(target)
    if not seq:
        print(f'  SKIPPING {target} — no sequence')
        continue
    emb = extract_protbert_embedding(seq, protbert_tokenizer, protbert_model, DEVICE)
    protbert_embeddings.append(emb)
    protbert_names.append(target)
    norm = float(np.linalg.norm(emb))
    print(f'  {target:8s}  shape={emb.shape}  norm={norm:.3f}')

protbert_arr = np.stack(protbert_embeddings, axis=0)  # (n_proteins, 1024)

np.save(PROTBERT_OUT_PATH, protbert_arr)
with open(PROTBERT_NAMES_PATH, 'w') as f:
    json.dump(protbert_names, f)

print(f'\nProtBERT embeddings saved: {PROTBERT_OUT_PATH}  shape={protbert_arr.shape}')
print(f'Names saved: {PROTBERT_NAMES_PATH}')

  PXR       shape=(1024,)  norm=3.190


  VDR       shape=(1024,)  norm=2.767


  FXR       shape=(1024,)  norm=3.282


  LXRa      shape=(1024,)  norm=3.239


  RXRa      shape=(1024,)  norm=3.119


  PPARg     shape=(1024,)  norm=3.273


  PPARa     shape=(1024,)  norm=2.796

ProtBERT embeddings saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_protbert_embeddings.npy  shape=(7, 1024)
Names saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_protbert_names.json


## 4. Summary

In [7]:
import os

print('=== Protein Embedding Summary ===')
print(f'\nTargets ({len(TARGET_ORDER)}): {TARGET_ORDER}')
print(f'\nESM-2 embeddings:')
print(f'  Shape:    {esm2_arr.shape}  (n_proteins x 320)')
print(f'  File:     {ESM2_OUT_PATH}')
print(f'  Size:     {os.path.getsize(ESM2_OUT_PATH) / 1024:.1f} KB')

print(f'\nProtBERT embeddings:')
print(f'  Shape:    {protbert_arr.shape}  (n_proteins x 1024)')
print(f'  File:     {PROTBERT_OUT_PATH}')
print(f'  Size:     {os.path.getsize(PROTBERT_OUT_PATH) / 1024:.1f} KB')

print(f'\nSequence cache:  {SEQ_CACHE}')
print(f'\nNext steps:')
print('  30_morgan_esm2_multinr.ipynb  — LGBM + Morgan + ESM-2 protein features')
print('  31_chemberta_esm2_multinr.ipynb — ChemBERTa + ESM-2')
print('  32_protbert_multinr.ipynb     — LGBM + Morgan + ProtBERT protein features')

=== Protein Embedding Summary ===

Targets (7): ['PXR', 'VDR', 'FXR', 'LXRa', 'RXRa', 'PPARg', 'PPARa']

ESM-2 embeddings:
  Shape:    (7, 320)  (n_proteins x 320)
  File:     D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_esm2_embeddings.npy
  Size:     8.9 KB

ProtBERT embeddings:
  Shape:    (7, 1024)  (n_proteins x 1024)
  File:     D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_protbert_embeddings.npy
  Size:     28.1 KB

Sequence cache:  D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\nr_sequences.json

Next steps:
  30_morgan_esm2_multinr.ipynb  — LGBM + Morgan + ESM-2 protein features
  31_chemberta_esm2_multinr.ipynb — ChemBERTa + ESM-2
  32_protbert_multinr.ipynb     — LGBM + Morgan + ProtBERT protein features
